# QoSBuddy — DSO1.3 Network Performance Benchmarking
**Member 5 | Production-Level Benchmarking Pipeline**

This notebook implements a **complete, end-to-end QoS benchmarking system** across 10 phases:

| Phase | Description |
|-------|-------------|
| 1 | Data Loading & Exploration |
| 2 | Data Preprocessing |
| 3 | QoS Score Engineering |
| 4 | Rule-Based Benchmarking |
| 5 | Machine Learning Classification |
| 6 | Category-Based Benchmarking |
| 7 | Visualisation |
| 8 | Insights Generation |
| 9 | Export Results |
| 10 | SHAP Explainability (Bonus) |


## Setup

In [ ]:
# Install optional packages if needed
# !pip install xgboost shap --quiet

# Import the benchmarking pipeline module
from benchmark_pipeline import (
    # Phase 1
    load_dataset, explore_dataset, plot_kpi_distributions, plot_correlation_matrix,
    # Phase 2
    preprocess,
    # Phase 3
    compute_qos_score,
    # Phase 4
    classify_performance_rule_based,
    # Phase 5
    train_and_evaluate_models, save_model,
    # Phase 6
    benchmark_by_category,
    # Phase 7
    plot_qos_score_distribution, plot_performance_class_distribution,
    plot_qos_by_load_level, plot_qos_by_mobility_speed,
    # Phase 8
    generate_insights,
    # Phase 9
    export_results,
    # Phase 10
    explain_with_shap, compare_rule_vs_ml,
    # Constants
    FEATURE_COLS,
)

print('Pipeline module loaded successfully.')


---
## Phase 1 — Data Loading & Exploration

Load the dataset (CSV or Parquet). If the file is not present, a **synthetic demo dataset** is generated automatically so every phase can be exercised end-to-end.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
# Update DATA_PATH to point to your real dataset when available.
# Supported formats: CSV (.csv) or Parquet (.parquet)
DATA_PATH = 'network_data.csv'   # e.g. 'data/telecom_qos.parquet'
N_ROWS    = None                  # Set to an integer to limit rows for large files

# Load dataset
df_raw = load_dataset(DATA_PATH, n_rows=N_ROWS)
df_raw.head()


In [ ]:
# Explore: head / info / describe / missing values
explore_dataset(df_raw)


In [ ]:
# KPI distribution histograms
plot_kpi_distributions(df_raw)


In [ ]:
# Pearson correlation heat-map
plot_correlation_matrix(df_raw)


---
## Phase 2 — Data Preprocessing

- Impute missing values with column medians (robust to outliers)
- Normalise QoS feature columns to **[0, 1]** using `MinMaxScaler`


In [ ]:
df, scaler = preprocess(df_raw)
norm_cols = [c for c in df.columns if c.endswith('_norm')]
print('Normalised columns:', norm_cols)
df[norm_cols].describe().T


---
## Phase 3 — QoS Score Engineering

Custom composite score in **[0, 1]** where *higher is better*:

```
QoS_score =  0.40 × norm_throughput
           + 0.20 × norm_sinr
           + 0.15 × (1 − norm_delay)
           + 0.15 × (1 − norm_jitter)
           + 0.10 × (1 − norm_packet_loss)
```


In [ ]:
df = compute_qos_score(df)
df[['throughput_mbps', 'delay_ms', 'jitter_ms', 'packet_loss_ratio', 'QoS_score']].describe().T


---
## Phase 4 — Rule-Based Benchmarking

| Class | Criteria |
|-------|----------|
| **GOOD** | throughput ≥ 20 Mbps **AND** delay ≤ 30 ms **AND** jitter ≤ 5 ms **AND** packet_loss ≤ 1 % |
| **POOR** | throughput ≤ 5 Mbps **OR** delay ≥ 100 ms **OR** jitter ≥ 20 ms **OR** packet_loss ≥ 5 % |
| **MEDIUM** | everything else |


In [ ]:
df = classify_performance_rule_based(df)
df['performance_class_rule'].value_counts()


---
## Phase 5 — Machine Learning Classification

Train **RandomForestClassifier** (and **XGBoostClassifier** if installed) to predict `performance_class_rule`.

Evaluation metrics: Accuracy, F1-score, Confusion Matrix.


In [ ]:
best_model, ml_results = train_and_evaluate_models(df, label_col='performance_class_rule')


In [ ]:
# Persist best model and scaler
save_model(best_model, scaler)


---
## Phase 6 — Category-Based Benchmarking

Group data by **load_level** and **mobility_speed** to compare performance across conditions.


In [ ]:
category_report = benchmark_by_category(df)
category_report


---
## Phase 7 — Visualisation


In [ ]:
# QoS score distribution
plot_qos_score_distribution(df)


In [ ]:
# Performance class distribution
plot_performance_class_distribution(df)


In [ ]:
# QoS score vs load level
plot_qos_by_load_level(df)


In [ ]:
# QoS score vs mobility speed
plot_qos_by_mobility_speed(df)


In [ ]:
# Full correlation heat-map (post-engineering)
plot_correlation_matrix(df)


---
## Phase 8 — Insights Generation


In [ ]:
generate_insights(df, category_report)


---
## Phase 9 — Export Results

Outputs are written to the `outputs/` directory:

- `dataset_with_qos_scores.csv` — full dataset with QoS scores and performance labels
- `benchmark_report_by_category.csv` — aggregated category-level benchmark


In [ ]:
export_results(df, category_report)


---
## Phase 10 — SHAP Explainability (Bonus)

> Install the `shap` package to enable this phase: `pip install shap`

- SHAP beeswarm and bar plots showing which features drive predictions
- Side-by-side comparison of rule-based vs ML labels


In [ ]:
# SHAP feature importance (requires 'shap' package)
explain_with_shap(best_model, df)


In [ ]:
# Rule-based vs ML prediction comparison
comparison = compare_rule_vs_ml(df, best_model)
comparison.head(10)


In [ ]:
# Agreement breakdown
if 'agreement' in comparison.columns:
    print('Agreement rate:', comparison['agreement'].mean())
    comparison.groupby(['performance_class_rule', 'performance_class_ml']).size().unstack(fill_value=0)


---
## Summary

| Output | Description |
|--------|-------------|
| `outputs/dataset_with_qos_scores.csv` | Full dataset enriched with QoS scores and labels |
| `outputs/benchmark_report_by_category.csv` | Aggregated KPIs per load_level × mobility_speed |
| `outputs/best_qos_model.pkl` | Serialised best ML classifier |
| `outputs/qos_scaler.pkl` | Fitted MinMaxScaler for inference |

All outputs are ready for integration into the QoSBuddy dashboard (Member 6).
